In [1]:
import numpy as np
import matplotlib.pyplot as plt
from math import factorial
from scipy.special import lpmv

In [2]:
def Y(m, l, phi, theta):
    m_abs = abs(m)
    x = np.cos(theta)
    
    if m >= 0:
        p_lm = lpmv(m, l, x)
    else:
        p_l_pos_m = lpmv(m_abs, l, x)
        factor = ((-1)**m_abs) * (factorial(l - m_abs) / factorial(l + m_abs))
        p_lm = factor * p_l_pos_m

    num = factorial(l - m)
    den = factorial(l + m)
    
    norm_theta = np.sqrt(((2 * l + 1) / 2.0) * (num / den))
    theta_lm = norm_theta * p_lm
   
    azimuthal_part = (1.0 / np.sqrt(2 * np.pi)) * np.exp(1j * m * phi)
    y_val = azimuthal_part * theta_lm
    
    return y_val

In [3]:
quality = 100
l = 3
m = 3


theta, phi = np.indices((quality, 2 * quality))
theta = (np.pi / quality) * theta
phi = (np.pi / quality) * phi

arr = Y(m, l, phi, theta)

In [4]:
import pyvista as pv
pv.set_jupyter_backend('trame')

def create_spherical_mesh(r_values):
    n_theta, n_phi = r_values.shape

    phi = np.linspace(0, 2 * np.pi, n_phi)
    theta = np.linspace(0, np.pi, n_theta)

    theta_grid, phi_grid = np.meshgrid(theta, phi, indexing='ij')

    x = r_values * np.sin(theta_grid) * np.cos(phi_grid)
    y = r_values * np.sin(theta_grid) * np.sin(phi_grid)
    z = r_values * np.cos(theta_grid)

    mesh = pv.StructuredGrid(x, y, z)
        
    return mesh


pl = pv.Plotter()

pl.add_mesh(create_spherical_mesh(np.abs(np.real(arr))))

pl.show()

Widget(value='<iframe src="http://localhost:39697/index.html?ui=P_0x79abab872690_0&reconnect=auto" class="pyvi…

In [5]:
import trimesh
import numpy as np

def load_stl_shape(file_path):
    """Loads an STL file and returns a trimesh object."""
    mesh = trimesh.load(file_path)
    
    if isinstance(mesh, trimesh.Scene):
        mesh = mesh.dump(concatenate=True)
        
    print(f"Loaded mesh with {len(mesh.vertices)} vertices and {len(mesh.faces)} faces.")
    return mesh

# 3. Calculate distance from center along a given vector
def get_distance_along_vector(mesh, center_coords, vector):

    origin = np.array(center_coords, dtype=np.float64)
    direction = np.array(vector, dtype=np.float64)
    
    # Normalize the direction vector (make its length exactly 1)
    norm = np.linalg.norm(direction)
    if norm == 0:
        raise ValueError("Direction vector cannot be a zero vector [0,0,0].")
    direction = direction / norm

    locations, index_ray, index_tri = mesh.ray.intersects_location(
        ray_origins=[origin],
        ray_directions=[direction]
    )

    if len(locations) == 0:
        return None 

    distances = np.linalg.norm(locations - origin, axis=1)

    closest_distance = np.min(distances)
    
    return closest_distance




my_shape = load_stl_shape('4.stl')
# Define the center (origin point)
center = [128, 127, 27] 
# center = [23.7, 9, 31.7] 
# center = [0, 0, 0]

Loaded mesh with 170 vertices and 336 faces.


In [6]:
quality = 100

theta, phi = np.indices((quality, 2 * quality))
theta = (np.pi / quality) * theta
phi = (np.pi / quality) * phi

vector_directions = np.stack([np.sin(theta) * np.cos(phi), np.sin(theta) * np.sin(phi), np.cos(theta)], axis = 2)

image = np.zeros((quality, 2 * quality))

for i in range(vector_directions.shape[0]):
    for j in range(vector_directions.shape[1]):
        image[i, j] = get_distance_along_vector(my_shape, center, vector_directions[i, j])

In [7]:
pl = pv.Plotter()

pl.add_mesh(create_spherical_mesh(image))

pl.show()

Widget(value='<iframe src="http://localhost:39697/index.html?ui=P_0x79ab3f0b6660_1&reconnect=auto" class="pyvi…

In [26]:
quality = 100
depth = 30
l = 1
m = 1


theta, phi = np.indices((quality, 2 * quality))
theta = (np.pi / quality) * theta
phi = (np.pi / quality) * phi

dOmega = np.sin(theta)

spherical_functions = dict()
for i in range (depth):
    for j in range(-i, i + 1):
        spherical_functions[(i, j)] = Y(j, i, phi, theta)

In [27]:
spherical_functions_k = dict()
for i in range(depth):
    for j in range(-i, i + 1):
        spherical_functions_k[(i, j)] = np.sum(image * np.conjugate(spherical_functions[(i, j)]) * dOmega) / np.sum(spherical_functions[(i, j)] * np.conjugate(spherical_functions[(i, j)]) * dOmega)

In [28]:
res = np.zeros((quality, 2 * quality))
for i in range(depth):
    for j in range(-i, i + 1):
        res = res + spherical_functions_k[(i, j)] * spherical_functions[(i, j)]

In [29]:
pl = pv.Plotter()

pl.add_mesh(create_spherical_mesh(np.real(res)))
pl.camera.roll += 135

pl.show() 

Widget(value='<iframe src="http://localhost:39697/index.html?ui=P_0x79ac03194980_9&reconnect=auto" class="pyvi…

In [30]:
pl = pv.Plotter()

pl.add_mesh(create_spherical_mesh(np.abs(np.real(spherical_functions[(3, 2)]))))

pl.show()

Widget(value='<iframe src="http://localhost:39697/index.html?ui=P_0x79ac03194fb0_10&reconnect=auto" class="pyv…

In [31]:
ijlist = []
for i in range(depth):
    for j in range(-i, i + 1):
        ijlist.append((i, j))

In [ ]:
import pyvista as pv
import numpy as np

num_slides = len(ijlist)
frames_data = []

for num_frame in range(num_slides):
    # Левый меш: сфера, которая постепенно увеличивается
    res = np.zeros((quality, 2 * quality))
    for k in range(num_frame + 1):
        i, j = ijlist[k]
        res = res + spherical_functions_k[(i, j)] * spherical_functions[(i, j)]
    left_mesh = create_spherical_mesh(np.real(res))
    
    # Правый меш: куб, который вращается
    right_mesh = create_spherical_mesh(np.abs(np.real(spherical_functions[ijlist[num_frame]])))
    
    frames_data.append((left_mesh, right_mesh))


fps = 30
output_filename = "split_screen_animation.mp4"

pl = pv.Plotter(shape=(1, 2), window_size=(1280, 720), off_screen=True)
pl.open_movie(output_filename, framerate=fps)

pl.subplot(0, 0)
pl.camera.roll += 135
pl.subplot(0, 1)
pl.camera.roll += 135

# Настройки времени (в секундах)
start_duration = 0.25  # Время показа первого слайда
end_duration = 0.04    # Время показа последнего слайда


for i, (mesh_l, mesh_r) in enumerate(frames_data):
    pl.subplot(0, 0)
    pl.add_mesh(mesh_l, name="left_object")
    # pl.camera_position = 'xyz' # Фиксируем камеру, чтобы она не прыгала
    pl.remove_actor(txt1)
    txt1 = pl.add_text(f"Number of basis functions {i + 1}", font_size=18)
    pl.reset_camera() 

    pl.subplot(0, 1)

    pl.add_mesh(mesh_r, name="right_object")
    l, m = ijlist[i]
    pl.remove_actor(txt2)
    txt2 = pl.add_text(f"L = {l}, M = {m}", font_size=18)
    # pl.camera_position = 'xyz'
    pl.reset_camera()


    if num_slides > 1:
        t = i / (num_slides - 1) # t меняется от 0.0 до 1.0
    else:
        t = 0.0
        
    current_duration_sec = start_duration * (1 - t) + end_duration * t
    
    # Переводим секунды в количество кадров видео
    # max(1, ...) гарантирует, что слайд покажется хотя бы на 1 кадр
    frames_to_write = max(1, int(current_duration_sec * fps))
    
    print(f"Слайд {i+1}/{num_slides}: {frames_to_write} кадров ({current_duration_sec:.2f} сек)")

    # Записываем нужное количество одинаковых кадров в видео
    for _ in range(frames_to_write):
        pl.write_frame()


pl.close()
print(f"Видео сохранено в {output_filename}")